In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("C:/Users/Sahil.s.Vernekar/OneDrive/Documents/ML/Churn/Dataset/train.csv")
df["num_support_tickets"] = np.random.poisson(lam=1.2, size=len(df))
custom_map ={
    'yes' : 1,
    'no' : 0
}
df['Churn_encoded'] = df['Churn'].apply(lambda x:1 if x=='Yes' else 0)
df.drop(columns =['Churn'], inplace = True)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print(df['InternetService'].nunique())

3


In [3]:
print(df.shape)
print(df.info())
print(df.sample(5))


(7043, 22)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customerID           7043 non-null   object 
 1   gender               7043 non-null   object 
 2   SeniorCitizen        7043 non-null   int64  
 3   Partner              7043 non-null   object 
 4   Dependents           7043 non-null   object 
 5   tenure               7043 non-null   int64  
 6   PhoneService         7043 non-null   object 
 7   MultipleLines        7043 non-null   object 
 8   InternetService      7043 non-null   object 
 9   OnlineSecurity       7043 non-null   object 
 10  OnlineBackup         7043 non-null   object 
 11  DeviceProtection     7043 non-null   object 
 12  TechSupport          7043 non-null   object 
 13  StreamingTV          7043 non-null   object 
 14  StreamingMovies      7043 non-null   object 
 15  Contract             7043 n

In [4]:
#DROP = ID
#ALL CATEGORICAL --> ONE HOT
#Standardize all NUMERIC COLS
#Chcek correlation
print(df['Contract'].nunique())
print(df['PaymentMethod'].nunique())

3
4


In [5]:
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

In [6]:
cols_to_drop = [cols for cols in df.columns if cols not in ['tenure', 'MonthlyCharges', 'TotalCharges', 'PaymentMethod', 'InternetService', 'Contract', 'num_support_tickets', 'Churn_encoded']]
imp_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'PaymentMethod', 'InternetService', 'Contract', 'num_support_tickets']

numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'num_support_tickets']
cat_cols = ['PaymentMethod', 'InternetService', 'Contract']
y = df['Churn_encoded']
df.drop(columns= cols_to_drop, inplace= True)
X = df.drop(columns=['Churn_encoded'])
print(numeric_cols)
print(cat_cols)

print(cat_cols)

['tenure', 'MonthlyCharges', 'TotalCharges', 'num_support_tickets']
['PaymentMethod', 'InternetService', 'Contract']
['PaymentMethod', 'InternetService', 'Contract']


In [7]:
print(X.info())
print(y.nunique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   tenure               7043 non-null   int64  
 1   InternetService      7043 non-null   object 
 2   Contract             7043 non-null   object 
 3   PaymentMethod        7043 non-null   object 
 4   MonthlyCharges       7043 non-null   float64
 5   TotalCharges         7032 non-null   float64
 6   num_support_tickets  7043 non-null   int32  
dtypes: float64(2), int32(1), int64(1), object(3)
memory usage: 357.8+ KB
None
2


In [8]:
correlation = df[numeric_cols].corr()
print(correlation)

                       tenure  MonthlyCharges  TotalCharges  \
tenure               1.000000        0.247900      0.825880   
MonthlyCharges       0.247900        1.000000      0.651065   
TotalCharges         0.825880        0.651065      1.000000   
num_support_tickets  0.004533       -0.003558      0.000820   

                     num_support_tickets  
tenure                          0.004533  
MonthlyCharges                 -0.003558  
TotalCharges                    0.000820  
num_support_tickets             1.000000  


In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy = 'median')),
    ('standardize', StandardScaler())  
])

In [10]:
OH_transformer = Pipeline(steps =[
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('OH', OneHotEncoder(handle_unknown="ignore"))
])

In [11]:
preprocessor = ColumnTransformer(transformers =[
    ('num', num_transformer, numeric_cols),
    ('OH', OH_transformer, cat_cols)
], remainder="drop")

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state = 42)
training_stats = {}

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'num_support_tickets']

cat_cols = ['PaymentMethod', 'InternetService', 'Contract']


import json
import numpy as np

def make_json_serializable(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj
with open("training_stats_v1.json", "w") as f:
    json.dump(training_stats, f, indent=4, default=make_json_serializable)


In [13]:
training_stats = {}

for col in num_cols:
    training_stats[col] = {
        "mean": float(X_train[col].mean()),
        "std": float(X_train[col].std()),
        "min": float(X_train[col].min()),
        "max": float(X_train[col].max()),
        "p25": float(X_train[col].quantile(0.25)),
        "p50": float(X_train[col].quantile(0.50)),
        "p75": float(X_train[col].quantile(0.75))
    }


In [14]:
training_stats[col] = {
    k: float(v)
    for k, v in X_train[col].value_counts(normalize=True).items()
}


In [15]:
with open("training_stats_v1.json", "w") as f:
    json.dump(training_stats, f, indent=4)


In [16]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42
)

In [17]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('xgb', xgb)
])

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('lr', LogisticRegression(
        max_iter=1000,
        solver="liblinear"
    ))
])

lr_pipeline.fit(X_train, y_train)
lr_preds = lr_pipeline.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_preds))

Logistic Regression Accuracy: 0.8007572172266919


In [19]:
param_grid = {
    "xgb__n_estimators": [100, 200],
    "xgb__max_depth": [3, 5],
    "xgb__learning_rate": [0.05, 0.1],
    "xgb__subsample": [0.8, 1.0],
    "xgb__colsample_bytree": [0.8, 1.0]
}

In [20]:
from sklearn.model_selection import RandomizedSearchCV

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=10,
    scoring="roc_auc",
    cv=3,
    random_state=42,
    n_jobs=-1
)

In [21]:
search.fit(X_train, y_train)

,estimator,"Pipeline(step...=None, ...))])"
,param_distributions,"{'xgb__colsample_bytree': [0.8, 1.0], 'xgb__learning_rate': [0.05, 0.1], 'xgb__max_depth': [3, 5], 'xgb__n_estimators': [100, 200], ...}"
,n_iter,10
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [29]:
best_params = search.best_params_
best_score = search.best_score_
best_model = search.best_estimator_

print(best_score)




0.8388760037685241


In [23]:
import joblib
joblib.dump(best_model, "churn_xgboost_v1.pkl")

['churn_xgboost_v1.pkl']

In [24]:
model_meta = {
    "model_name": "churn_xgboost",
    "version": "v1",
    "algorithm": "XGBoost",
    "features": imp_cols,
    "roc_auc": search.best_score_
}

with open("model_meta_v1.json", "w") as f:
    json.dump(model_meta, f, indent=4)


In [25]:
X_final = pd.concat([X_train, X_test])
y_final = pd.concat([y_train, y_test])

clean_params = {
    k.replace("xgb__", ""): v
    for k, v in best_params.items()
}


final_model = XGBClassifier(**clean_params,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42)
final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("xgb", final_model)
])

final_pipeline.fit(X_final, y_final)


,steps,"[('preprocessor', ...), ('xgb', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('OH', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [31]:
y_pred = final_pipeline.predict(X_test)

In [26]:
final_pipeline.named_steps["xgb"].get_params()


{'objective': 'binary:logistic',
 'base_score': None,
 'booster': None,
 'callbacks': None,
 'colsample_bylevel': None,
 'colsample_bynode': None,
 'colsample_bytree': 0.8,
 'device': None,
 'early_stopping_rounds': None,
 'enable_categorical': False,
 'eval_metric': 'auc',
 'feature_types': None,
 'feature_weights': None,
 'gamma': None,
 'grow_policy': None,
 'importance_type': None,
 'interaction_constraints': None,
 'learning_rate': 0.05,
 'max_bin': None,
 'max_cat_threshold': None,
 'max_cat_to_onehot': None,
 'max_delta_step': None,
 'max_depth': 3,
 'max_leaves': None,
 'min_child_weight': None,
 'missing': nan,
 'monotone_constraints': None,
 'multi_strategy': None,
 'n_estimators': 100,
 'n_jobs': None,
 'num_parallel_tree': None,
 'random_state': 42,
 'reg_alpha': None,
 'reg_lambda': None,
 'sampling_method': None,
 'scale_pos_weight': None,
 'subsample': 0.8,
 'tree_method': None,
 'validate_parameters': None,
 'verbosity': None}

In [27]:
test_row = pd.DataFrame([{
    "tenure": 10,
    "MonthlyCharges": 70,
    "TotalCharges": 700,
    "Contract": "One year",
    "PaymentMethod": "Mailed check",
    "InternetService": "DSL",
    "random_col": 999,
    'num_support_tickets':3
}])

final_pipeline.predict_proba(test_row)


array([[0.8847722 , 0.11522779]], dtype=float32)

In [28]:
import joblib

joblib.dump(preprocessor, 'preprocessor.plk')
joblib.dump(final_model, 'model.plk')
joblib.dump(final_pipeline, 'pipeline.plk')

['pipeline.plk']

In [33]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [34]:
stats = {"accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred)}
print(stats)

{'accuracy': 0.8059630856601988, 'f1': 0.5980392156862745, 'precision': 0.6838565022421524, 'recall': 0.5313588850174216}
